Heat Flux estimation via lumped capacity approach
Assumptions:
- In each part is constant (and equally distributed) tempreature before and after firing
- The parts are each perfectly insulated from the outside and each other
- The heat flux can be averaged and does not depend on the part temperature (especially not hot wall temp)
- Heat capacity of CCZ is known
- Exact firing duration is known

In [4]:
heat_capacity_ccz = 385 # kJ/kg*K
aerospike_weight = 6.331 # kg
straight_weight = 5.013 # kg
wall_weight = 10.394 # kg
aerospike_area_total = 34263.197 # mm^2
aerospike_area_cc = 21362.83 # mm^2
straight_area = 21362.83 # mm^2
wall_area = 23706.418 # mm^2


firing_duration = 4 # s
# Issue here: We had an actual firing for 1.5s. After that, with the purge active, the firing continued for further .5s. The CC pressure starts dropping and roughly hits 0 barg after 4s total
core_start_temp = -95 # °C
core_final_temp = 70 # °C

aerospike_start_temp = -83.6 # °C
aerospike_final_temp = 216.3 # °C

wall_start_temp = -75 # °C
wall_final_temp = 70 # °C

Q_tot_core = straight_weight * heat_capacity_ccz * (core_final_temp - core_start_temp)
Q_tot_aerospike = aerospike_weight * heat_capacity_ccz * (aerospike_final_temp - aerospike_start_temp)
Q_tot_wall = wall_weight * heat_capacity_ccz * (wall_final_temp - wall_start_temp)

print("Total Heat Energy in the core: ", Q_tot_core, "J" )
print("Total Heat Energy in the aerospike: ", Q_tot_aerospike, "J")
print("Total Heat Energy in the wall: ", Q_tot_wall, "J")
print("\n", "---------------------------------------", "\n")

# Calculate Heat Flux
q_core = Q_tot_core/(straight_area*firing_duration)
q_aerospike_cc = Q_tot_aerospike/(aerospike_area_cc*firing_duration)
q_aerospike_tot = Q_tot_aerospike/(aerospike_area_total*firing_duration)
q_wall = Q_tot_wall/(wall_area*firing_duration)

print("Heat Flux to core: ", q_core, "MW/m^2" )
print("Heat Flux to aerospike (only CC): ", q_aerospike_cc, " MW/m^2")
print("Heat Flux to aerospike (total): ", q_aerospike_tot, " MW/m^2")
print("Heat Flux to wall: ", q_wall, " MW/m^2")

Total Heat Energy in the core:  318450.82499999995 J
Total Heat Energy in the aerospike:  730986.7564999999 J
Total Heat Energy in the wall:  580245.05 J

 --------------------------------------- 

Heat Flux to core:  3.7266928702798263 MW/m^2
Heat Flux to aerospike (only CC):  8.554423225995805  MW/m^2
Heat Flux to aerospike (total):  5.333614639783905  MW/m^2
Heat Flux to wall:  6.1190713206862375  MW/m^2


In [2]:
"""
Perforated Plate Air Mass Flow Estimator
=========================================
Estimates the total air mass flow through a perforated surface given:
  - Upstream static pressure (gauge or absolute)
  - Upstream temperature
  - Hole geometry and discharge coefficient (calibrated with water)
  - Number of holes

Handles both subsonic (with compressibility correction factor Y)
and choked (sonic) conditions per-hole.

References:
  - ISO 5167 / ASME MFC-3M orifice equations
  - Compressible flow through orifices (Anderson, "Modern Compressible Flow")
"""

import math

# ──────────────────────────────────────────────
# Constants
# ──────────────────────────────────────────────
GAMMA = 1.4          # Ratio of specific heats for air
R_AIR = 287.05       # J/(kg·K), specific gas constant for air
P_AMB = 101325.0     # Pa, standard ambient (downstream) pressure

# Critical pressure ratio for choked flow
# P1/P2 > PR_CRIT  →  flow is choked
PR_CRIT = (2.0 / (GAMMA + 1)) ** (-GAMMA / (GAMMA - 1))   # ≈ 1.893




Cd          = 0.88 # get_float("  Discharge coefficient Cd           [–]   : ")
d_mm        = 0.3 # get_float("  Hole diameter                       [mm]  : ")
n_holes     = 96*3 # dumb assumption just for my interest, to see how much n2 would flow through all orifices
# get_int  ("  Number of holes                     [–]   : ")
P1_kPa      = 200 # get_float("  Upstream GAUGE pressure             [kPa] : ")
T1_C        = 20 # get_float("  Upstream temperature                [°C]  (default 20): ", default=20.0)
beta        = 0 # get_float("  Diameter ratio β = d/D (0 if open)  [–]   (default 0): ", default=0.0)



# ──────────────────────────────────────────────
# Core functions
# ──────────────────────────────────────────────

def upstream_density(P1_abs: float, T1: float) -> float:
    """Ideal-gas density at upstream conditions [kg/m³]."""
    return P1_abs / (R_AIR * T1)


def expansion_factor_Y(P1_abs: float, P2_abs: float, beta: float = 0.0) -> float:
    """
    Compressibility expansion factor Y for a subsonic orifice.

    For thin perforated plates with no surrounding pipe, beta → 0,
    which simplifies the expression considerably.

    Parameters
    ----------
    P1_abs : upstream absolute pressure [Pa]
    P2_abs : downstream absolute pressure [Pa]
    beta   : diameter ratio d/D; use 0 for free-discharge holes
    """
    r = P2_abs / P1_abs          # pressure ratio  (0 < r ≤ 1)
    g = GAMMA

    # Numerator factor
    num = (r ** (2.0 / g)) - (r ** ((g + 1.0) / g))
    num *= g / (g - 1.0)

    # Denominator factor
    den = 1.0 - r

    # Beta correction (≈1 for beta=0)
    beta4 = beta ** 4
    beta_corr = math.sqrt((1.0 - beta4) / (1.0 - beta4 * r ** (2.0 / g)))

    Y = math.sqrt(num / den) * beta_corr
    return Y


def mass_flow_single_hole(
    Cd: float,
    d_hole: float,
    P1_abs: float,
    T1: float,
    P2_abs: float = P_AMB,
    beta: float = 0.0,
) -> tuple[float, bool]:
    """
    Mass flow rate through a single orifice hole [kg/s].

    Returns (mdot, is_choked).

    Parameters
    ----------
    Cd      : discharge coefficient  [-]
    d_hole  : hole diameter          [m]
    P1_abs  : upstream absolute pressure  [Pa]
    T1      : upstream temperature        [K]
    P2_abs  : downstream absolute pressure [Pa]  (default: ambient)
    beta    : diameter ratio d/D for the orifice  [-]  (default: 0)
    """
    A = math.pi / 4.0 * d_hole ** 2      # hole area [m²]
    rho1 = upstream_density(P1_abs, T1)

    choked = (P1_abs / P2_abs) >= PR_CRIT

    if choked:
        # Isentropic choked-flow formula
        # mdot = Cd * A * P1 * sqrt(gamma / (R*T1)) * (2/(gamma+1))^((gamma+1)/(2*(gamma-1)))
        factor = (2.0 / (GAMMA + 1.0)) ** ((GAMMA + 1.0) / (2.0 * (GAMMA - 1.0)))
        mdot = Cd * A * P1_abs * math.sqrt(GAMMA / (R_AIR * T1)) * factor
    else:
        # Subsonic: incompressible orifice equation × expansion factor Y
        dP = P1_abs - P2_abs
        Y = expansion_factor_Y(P1_abs, P2_abs, beta)
        mdot = Cd * Y * A * math.sqrt(2.0 * rho1 * dP)

    return mdot, choked


def total_mass_flow(
    Cd: float,
    d_hole: float,
    n_holes: int,
    P1_gauge: float,
    T1_celsius: float,
    P2_abs: float = P_AMB,
    beta: float = 0.0,
) -> dict:
    """
    Total air mass flow through all holes [kg/s].

    Parameters
    ----------
    Cd          : discharge coefficient                [-]
    d_hole      : hole diameter                        [m]
    n_holes     : number of holes                      [-]
    P1_gauge    : upstream gauge pressure              [Pa]
    T1_celsius  : upstream temperature                 [°C]
    P2_abs      : downstream absolute pressure         [Pa]
    beta        : orifice diameter ratio d/D           [-]

    Returns a dict with all relevant computed quantities.
    """
    P1_abs = P1_gauge + P2_abs
    T1 = T1_celsius + 273.15

    mdot_per_hole, choked = mass_flow_single_hole(
        Cd, d_hole, P1_abs, T1, P2_abs, beta
    )
    mdot_total = mdot_per_hole * n_holes

    rho1 = upstream_density(P1_abs, T1)
    A_hole = math.pi / 4.0 * d_hole ** 2
    A_total = A_hole * n_holes

    # Volumetric flow at upstream conditions and at ambient conditions
    Qvol_upstream = mdot_total / rho1
    rho_amb = upstream_density(P2_abs, T1)   # same T assumed at exit
    Qvol_ambient = mdot_total / rho_amb

    return {
        "mdot_per_hole_kg_s":  mdot_per_hole,
        "mdot_total_kg_s":     mdot_total,
        "mdot_total_g_s":      mdot_total * 1e3,
        "Qvol_upstream_m3s":   Qvol_upstream,
        "Qvol_ambient_m3s":    Qvol_ambient,
        "Qvol_ambient_Lmin":   Qvol_ambient * 1e3 * 60.0,
        "choked":              choked,
        "P1_abs_Pa":           P1_abs,
        "rho1_kg_m3":          rho1,
        "A_hole_m2":           A_hole,
        "A_total_m2":          A_total,
        "pressure_ratio":      P1_abs / P2_abs,
        "critical_PR":         PR_CRIT,
    }


# ──────────────────────────────────────────────
# Pretty-print helper
# ──────────────────────────────────────────────

def print_results(r: dict, Cd: float, d_mm: float, n: int,
                  P1_gauge_Pa: float, T1_C: float) -> None:
    sep = "─" * 52
    print(f"\n{'═'*52}")
    print(f"  Perforated Plate Air Flow Estimate")
    print(f"{'═'*52}")
    print(f"  Inputs")
    print(sep)
    print(f"  Discharge coefficient Cd  : {Cd:.4f}")
    print(f"  Hole diameter             : {d_mm:.2f} mm")
    print(f"  Number of holes           : {n}")
    print(f"  Upstream gauge pressure   : {P1_gauge_Pa/1e3:.3f} kPa  "
          f"({P1_gauge_Pa:.1f} Pa)")
    print(f"  Upstream temperature      : {T1_C:.1f} °C")
    print(f"  Downstream pressure       : {P_AMB/1e3:.3f} kPa (ambient)")
    print(f"\n  Flow regime")
    print(sep)
    print(f"  Pressure ratio P1/P2      : {r['pressure_ratio']:.4f}")
    print(f"  Critical PR (choke limit) : {r['critical_PR']:.4f}")
    print(f"  Flow condition            : {'⚠ CHOKED (sonic)' if r['choked'] else 'Subsonic'}")
    print(f"\n  Results")
    print(sep)
    print(f"  Mass flow per hole        : {r['mdot_per_hole_kg_s']*1e6:.4f} mg/s"
          f"  ({r['mdot_per_hole_kg_s']*1e3:.6f} g/s)")
    print(f"  Total mass flow           : {r['mdot_total_kg_s']*1e3:.4f} g/s"
          f"  ({r['mdot_total_kg_s']:.6f} kg/s)")
    print(f"  Vol. flow @ upstream ρ    : {r['Qvol_upstream_m3s']*1e6:.4f} cm³/s"
          f"  ({r['Qvol_upstream_m3s']*1e3*60:.3f} L/min)")
    print(f"  Vol. flow @ ambient ρ     : {r['Qvol_ambient_m3s']*1e6:.4f} cm³/s"
          f"  ({r['Qvol_ambient_Lmin']:.3f} L/min)")
    print(f"  Upstream air density ρ₁   : {r['rho1_kg_m3']:.4f} kg/m³")
    print(f"  Total hole area           : {r['A_total_m2']*1e6:.4f} mm²")
    print(f"{'═'*52}\n")


# ──────────────────────────────────────────────
# Actual Calculation
# ──────────────────────────────────────────────



r = total_mass_flow(
    Cd      = Cd,
    d_hole  = d_mm / 1e3,
    n_holes = n_holes,
    P1_gauge= P1_kPa * 1e3,
    T1_celsius = T1_C,
    beta    = beta,
)

print_results(r, Cd, d_mm, n_holes, P1_kPa * 1e3, T1_C)


# ──────────────────────────────────────────────
# Example / demo  (runs if no interactive input)
# ──────────────────────────────────────────────

def run_demo():
    print("\n  ── Demo: 3 example cases ──")

    cases = [
        dict(label="Low ΔP (subsonic)",
             Cd=0.65, d_mm=3.0, n=12, P1_kPa=1.0,  T1_C=20.0),
        dict(label="Moderate ΔP (subsonic)",
             Cd=0.65, d_mm=3.0, n=12, P1_kPa=10.0, T1_C=20.0),
        dict(label="High ΔP (choked)",
             Cd=0.65, d_mm=3.0, n=12, P1_kPa=100.0, T1_C=20.0),
    ]

    for c in cases:
        print(f"\n  Case: {c['label']}")
        r = total_mass_flow(
            Cd=c["Cd"], d_hole=c["d_mm"]/1e3, n_holes=c["n"],
            P1_gauge=c["P1_kPa"]*1e3, T1_celsius=c["T1_C"],
        )
        regime = "CHOKED" if r["choked"] else "subsonic"
        print(f"    P1_gauge={c['P1_kPa']} kPa | PR={r['pressure_ratio']:.3f} | "
              f"{regime} | "
              f"ṁ_total = {r['mdot_total_g_s']:.4f} g/s  "
              f"({r['Qvol_ambient_Lmin']:.2f} L/min @ ambient)")



════════════════════════════════════════════════════
  Perforated Plate Air Flow Estimate
════════════════════════════════════════════════════
  Inputs
────────────────────────────────────────────────────
  Discharge coefficient Cd  : 0.8800
  Hole diameter             : 0.30 mm
  Number of holes           : 192
  Upstream gauge pressure   : 200.000 kPa  (200000.0 Pa)
  Upstream temperature      : 20.0 °C
  Downstream pressure       : 101.325 kPa (ambient)

  Flow regime
────────────────────────────────────────────────────
  Pressure ratio P1/P2      : 2.9738
  Critical PR (choke limit) : 1.8929
  Flow condition            : ⚠ CHOKED (sonic)

  Results
────────────────────────────────────────────────────
  Mass flow per hole        : 44.2432 mg/s  (0.044243 g/s)
  Total mass flow           : 8.4947 g/s  (0.008495 kg/s)
  Vol. flow @ upstream ρ    : 2372.2491 cm³/s  (142.335 L/min)
  Vol. flow @ ambient ρ     : 7054.7049 cm³/s  (423.282 L/min)
  Upstream air density ρ₁   : 3.5809 kg/m³